In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import re
from datetime import datetime
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import LinearSVC, SVC
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, accuracy_score, auc, confusion_matrix, f1_score, precision_score, recall_score, roc_curve
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import stats
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df = pd.read_csv('data_football_ratings.csv')

In [3]:
# Mantener solo las filas donde is_human sea diferente de 1
df = df[df["is_human"] != 1]

In [4]:
df_defensa = df[df["rater"]=='WhoScored']
df_defensa.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21354 entries, 1 to 50651
Data columns (total 63 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   competition             21354 non-null  object 
 1   date                    21354 non-null  object 
 2   match                   21354 non-null  object 
 3   team                    21354 non-null  object 
 4   pos                     21354 non-null  object 
 5   pos_role                21354 non-null  object 
 6   player                  21354 non-null  object 
 7   rater                   21354 non-null  object 
 8   is_human                21354 non-null  int64  
 9   original_rating         21354 non-null  float64
 10  goals                   21354 non-null  int64  
 11  assists                 21354 non-null  int64  
 12  shots_ontarget          21354 non-null  int64  
 13  shots_offtarget         21354 non-null  int64  
 14  shotsblocked            21354 non-null  int

In [5]:
# Filtrar solo filas donde pos = 'GK'
df_defensa = df_defensa[df_defensa["pos"] == "DF"].copy()

# Ver las primeras filas
df_defensa.head()

,competition,date,match,team,pos,pos_role,player,rater,is_human,original_rating,...,betweenness_centrality,closeness_centrality,flow_centrality,flow_success,betweenness2goals,win,lost,is_home_team,minutesPlayed,game_duration
1,Euro 2016,10/06/2016,"France - Romania, 2 - 1",Romania,DF,DC,Dragos Grigore,WhoScored,0,6.56,...,0.143055,0.603571,0.304348,0.000000,0.0,0,1,0,90,90
11,Euro 2016,10/06/2016,"France - Romania, 2 - 1",Romania,DF,DL,Razvan Rat,WhoScored,0,6.38,...,0.321679,0.754464,0.478261,0.000000,0.0,0,1,0,90,90
17,Euro 2016,10/06/2016,"France - Romania, 2 - 1",France,DF,DC,Laurent Koscielny,WhoScored,0,6.80,...,0.124054,0.574830,0.315068,2.917816,0.0,1,0,1,90,90
35,Euro 2016,10/06/2016,"France - Romania, 2 - 1",Romania,DF,DC,Vlad Chiriches,WhoScored,0,6.81,...,0.200383,0.635338,0.275362,0.606906,0.0,0,1,0,90,90
38,Euro 2016,10/06/2016,"France - Romania, 2 - 1",Romania,DF,DR,Cristian Sapunaru,WhoScored,0,6.59,...,0.174008,0.603571,0.260870,0.553696,0.0,0,1,0,90,90


In [6]:
df_defensa.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5898 entries, 1 to 50636
Data columns (total 63 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   competition             5898 non-null   object 
 1   date                    5898 non-null   object 
 2   match                   5898 non-null   object 
 3   team                    5898 non-null   object 
 4   pos                     5898 non-null   object 
 5   pos_role                5898 non-null   object 
 6   player                  5898 non-null   object 
 7   rater                   5898 non-null   object 
 8   is_human                5898 non-null   int64  
 9   original_rating         5898 non-null   float64
 10  goals                   5898 non-null   int64  
 11  assists                 5898 non-null   int64  
 12  shots_ontarget          5898 non-null   int64  
 13  shots_offtarget         5898 non-null   int64  
 14  shotsblocked            5898 non-null   int6

In [7]:
cols_to_drop = [
    "competition", "date", "match", "team", "player", "rater", "is_human",
    "degree_centrality", "betweenness_centrality", "closeness_centrality",
    "flow_centrality", "flow_success", "betweenness2goals","pos_role","pos"
]

In [8]:
df_defensa = df_defensa.drop(columns= cols_to_drop)

In [9]:
df_defensa.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5898 entries, 1 to 50636
Data columns (total 48 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   original_rating   5898 non-null   float64
 1   goals             5898 non-null   int64  
 2   assists           5898 non-null   int64  
 3   shots_ontarget    5898 non-null   int64  
 4   shots_offtarget   5898 non-null   int64  
 5   shotsblocked      5898 non-null   int64  
 6   chances2score     5898 non-null   int64  
 7   drib_success      5898 non-null   int64  
 8   drib_unsuccess    5898 non-null   int64  
 9   keypasses         5898 non-null   int64  
 10  touches           5898 non-null   int64  
 11  passes_acc        5898 non-null   int64  
 12  passes_inacc      5898 non-null   int64  
 13  crosses_acc       5898 non-null   int64  
 14  crosses_inacc     5898 non-null   int64  
 15  lballs_acc        5898 non-null   int64  
 16  lballs_inacc      5898 non-null   int64  
 17 

In [10]:
corrs = df_defensa.corr()['original_rating'].sort_values(ascending=False)
print(corrs)

original_rating     1.000000
win                 0.435072
aerials_w           0.365598
goals               0.300110
touches             0.258470
shots_ontarget      0.231033
assists             0.206270
grduels_w           0.202241
clearances          0.201515
tackles             0.196895
minutesPlayed       0.185093
passes_acc          0.158998
drib_success        0.135091
poss_lost           0.131633
interceptions       0.128962
chances2score       0.122906
aerials_l           0.120913
is_home_team        0.116410
countattack         0.114329
keypasses           0.111910
stop_shots          0.111831
lballs_acc          0.095827
wasfouled           0.088173
shots_offtarget     0.081925
tballs_acc          0.076806
passes_inacc        0.068966
lballs_inacc        0.062582
crosses_acc         0.062398
game_duration       0.037091
shotsblocked        0.024024
tballs_inacc        0.017228
offsides            0.013286
crosses_inacc      -0.006235
drib_unsuccess     -0.018270
missed_penalti

In [11]:
# Separar X y y
X = df_defensa.drop(columns=["original_rating"])
y = df_defensa["original_rating"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.4 MB/s eta 0:00:00


In [19]:
from catboost import CatBoostRegressor

# Entrenar modelo
catboost = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
catboost.fit(X_train, y_train)

# Predecir en test
y_pred = catboost.predict(X_test)

# Calcular métricas
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"CatBoost R²: {r2:.4f}")
print(f"CatBoost RMSE: {rmse:.4f}")

CatBoost R²: 0.5928
CatBoost RMSE: 0.4138


In [14]:
import pandas as pd

# Obtener importancias
importancias = catboost.get_feature_importance(prettified=True)

# Si quieres sólo las más importantes, ordena y selecciona las top N
top_features = importancias.sort_values('Importances', ascending=False)

print(top_features)  # Muestra todas ordenadas

# Opcional: mostrar sólo las 10 primeras
print(top_features.head(20))

          Feature Id  Importances
0               lost    18.049804
1          aerials_w    15.081388
2                win     8.747628
3              goals     6.656407
4          grduels_w     5.105523
5         clearances     4.878109
6            touches     3.793023
7            assists     3.352218
8      interceptions     2.775610
9       drib_success     2.766788
10        stop_shots     2.614153
11           tackles     2.576778
12            ycards     1.993390
13         keypasses     1.820238
14         poss_lost     1.773771
15      dangmistakes     1.560377
16             fouls     1.426005
17        passes_acc     1.420206
18        lballs_acc     1.400534
19    shots_ontarget     1.333063
20       crosses_acc     1.096288
21      lballs_inacc     1.093479
22         grduels_l     0.964355
23      passes_inacc     0.846399
24            rcards     0.832703
25     minutesPlayed     0.680281
26          owngoals     0.637320
27       countattack     0.632337
28     crosses

In [15]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Lista de modelos a evaluar
modelos = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5),
    "RandomForest": RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42),
    "SVR": SVR(kernel="rbf", C=10, epsilon=0.1),
    "KNN": KNeighborsRegressor(n_neighbors=5)
}

# Entrenar y evaluar cada modelo
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"{nombre} -> R²: {r2:.4f}, RMSE: {rmse:.4f}")

LinearRegression -> R²: 0.6074, RMSE: 0.4063
Ridge -> R²: 0.6076, RMSE: 0.4062
Lasso -> R²: 0.5693, RMSE: 0.4256
ElasticNet -> R²: 0.5816, RMSE: 0.4194
RandomForest -> R²: 0.5248, RMSE: 0.4470
GradientBoosting -> R²: 0.5742, RMSE: 0.4232
SVR -> R²: 0.5253, RMSE: 0.4468
KNN -> R²: 0.1465, RMSE: 0.5991


In [16]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

# Escalar las variables predictoras (importante para redes)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Definir modelo simple de red neuronal
model_nn = Sequential()
model_nn.add(Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)))
model_nn.add(Dropout(0.2))
model_nn.add(Dense(32, activation='relu'))
model_nn.add(Dense(1))  # Capa de salida para regresión

# Compilar modelo
model_nn.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

# Entrenar modelo (puedes bajar epochs para acelerar)
history = model_nn.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=1,
    callbacks=[]  # Puedes añadir EarlyStopping para evitar sobreentrenar
)

# Predecir en test
y_pred_nn = model_nn.predict(X_test_scaled).flatten()

# Calcular métricas
r2 = r2_score(y_test, y_pred_nn)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_nn))

print(f"Red neuronal R²: {r2:.4f}")
print(f"Red neuronal RMSE: {rmse:.4f}")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
118/118 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - loss: 32.7509 - val_loss: 2.4653
Epoch 2/50
118/118 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 2.5388 - val_loss: 1.2716
Epoch 3/50
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.6356 - val_loss: 0.9985
Epoch 4/50
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.4510 - val_loss: 0.8386
Epoch 5/50
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.2438 - val_loss: 0.7698
Epoch 6/50
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.1998 - val_loss: 0.6578
Epoch 7/50
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.9994 - val_loss: 0.5898
Epoch 8/50
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8678 - val_loss: 0.5432
Epoch 9/50
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8088 - val_loss: 0.4959
Epoch 10/50
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.7723 - val_loss: 0.4720
Epoch 11/50
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6951 - val_loss: 0.4432
Epoch 12/50
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/st

In [20]:
import pandas as pd

# Suponiendo que y_test es un array/pd.Series y y_pred es un array
comparacion = pd.DataFrame({
    'Original': y_test,
    'Prediccion': y_pred
})

# Mostrar las primeras filas para ver los valores
print(comparacion.head(20))

# Si quieres también ver el error absoluto de cada predicción:
comparacion['Error absoluto'] = (comparacion['Original'] - comparacion['Prediccion']).abs()
print(comparacion.head(20))

       Original  Prediccion
48623      7.80    7.114785
17528      7.17    7.804251
15753      7.97    7.511890
31349      7.15    6.245779
44398      5.40    6.258384
12447      5.98    5.992841
20559      6.48    6.598490
10910      6.88    6.288354
49677      6.59    6.701498
9737       6.92    6.697075
6140       8.23    7.436170
49205      6.88    7.628077
28237      6.26    6.199722
28092      6.88    6.593781
11706      7.89    7.132113
24829      6.20    6.478382
40763      7.11    6.645778
16840      6.94    7.135037
33250      7.06    7.395344
12873      6.04    6.660012
       Original  Prediccion  Error absoluto
48623      7.80    7.114785        0.685215
17528      7.17    7.804251        0.634251
15753      7.97    7.511890        0.458110
31349      7.15    6.245779        0.904221
44398      5.40    6.258384        0.858384
12447      5.98    5.992841        0.012841
20559      6.48    6.598490        0.118490
10910      6.88    6.288354        0.591646
49677      6.59 

In [21]:
import numpy as np

# Asumiendo que tienes win (1/0) y lost (1/0)
# Empate donde ni win ni lost son 1
df_defensa['result'] = np.where(df_defensa['win'] == 1, 'victory',
                 np.where(df_defensa['lost'] == 1, 'defeat', 'draw'))

# Luego elimina las columnas originales
df_defensa = df_defensa.drop(columns=['win', 'lost'])

In [22]:
df_defensa['duelos_ganados'] = df_defensa['aerials_w'] + df_defensa['grduels_w']
df_defensa = df_defensa.drop(columns=['aerials_w', 'grduels_w'])

In [23]:
# Lista de columnas que quieres conservar
cols = [
    "goals", "touches", "assists", "result", "duelos_ganados", "poss_lost",
    "passes_acc", "minutesPlayed", "interceptions", "tackles",
    "ycards", "rcards", "clearances", "stop_shots",
    "dangmistakes", "fouls", "original_rating"
]

# Crear nuevo DataFrame solo con esas columnas
df_defensa_filtrado = df_defensa[cols]


In [28]:
df_defensa_filtrado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5898 entries, 1 to 50636
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   goals            5898 non-null   int64  
 1   touches          5898 non-null   int64  
 2   assists          5898 non-null   int64  
 3   result           5898 non-null   object 
 4   duelos_ganados   5898 non-null   int64  
 5   poss_lost        5898 non-null   int64  
 6   passes_acc       5898 non-null   int64  
 7   minutesPlayed    5898 non-null   int64  
 8   interceptions    5898 non-null   int64  
 9   tackles          5898 non-null   int64  
 10  ycards           5898 non-null   int64  
 11  rcards           5898 non-null   int64  
 12  clearances       5898 non-null   int64  
 13  stop_shots       5898 non-null   int64  
 14  dangmistakes     5898 non-null   int64  
 15  fouls            5898 non-null   int64  
 16  original_rating  5898 non-null   float64
dtypes: float64(1), int

In [29]:
df_defensa_filtrado.head()

,goals,touches,assists,result,duelos_ganados,poss_lost,passes_acc,minutesPlayed,interceptions,tackles,ycards,rcards,clearances,stop_shots,dangmistakes,fouls,original_rating
1,0,34,0,defeat,0,11,20,90,3,0,0,0,8,0,0,0,6.56
11,0,80,0,defeat,6,21,31,90,2,2,1,0,1,0,0,1,6.38
17,0,55,0,victory,4,10,37,90,5,0,0,0,5,0,0,0,6.80
35,0,38,0,defeat,5,6,20,90,2,2,1,0,2,1,0,1,6.81
38,0,45,0,defeat,2,14,16,90,4,1,0,0,2,1,0,1,6.59


In [39]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Separar X e y
X = df_defensa_filtrado.drop(columns=['original_rating'])
y = df_defensa_filtrado['original_rating']

# Dividir en train y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Definir variables categóricas si las hay (ajustar según tu dataset)
cat_features = ['result']  # ejemplo

# Obtener índices de variables categóricas
cat_features_idx = [X_train.columns.get_loc(col) for col in cat_features if col in X_train.columns]

# Inicializar y entrenar modelo CatBoost
model = CatBoostRegressor(iterations=1000,learning_rate=0.08, depth=6, random_seed=42,verbose=0, l2_leaf_reg= 6,min_data_in_leaf=1,bagging_temperature=0)
model.fit(X_train, y_train, cat_features=cat_features_idx)

# Predecir y evaluar
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"CatBoost R²: {r2:.4f}")
print(f"CatBoost RMSE: {rmse:.4f}")

CatBoost R²: 0.5449
CatBoost RMSE: 0.4375


In [40]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import itertools

# --- Datos (como ya tienes) ---
X = df_defensa_filtrado.drop(columns=['original_rating'])
y = df_defensa_filtrado['original_rating']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Split de validación para early stopping
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# Variables categóricas (ajusta la lista si hay más)
cat_features = ['result']
cat_features_idx = [X_tr.columns.get_loc(c) for c in cat_features if c in X_tr.columns]

# --- Espacio pequeño de búsqueda (rápido) ---
grid = {
    "depth": [6, 8],
    "learning_rate": [0.03, 0.05, 0.08],
    "l2_leaf_reg": [3, 6, 10],
    "bagging_temperature": [0, 1],
    "min_data_in_leaf": [1, 10]
}
param_sets = list(itertools.product(
    grid["depth"], grid["learning_rate"], grid["l2_leaf_reg"],
    grid["bagging_temperature"], grid["min_data_in_leaf"]
))

best_r2 = -np.inf
best_params = None
best_model = None

# --- Grid search manual ---
for depth, lr, l2, bt, min_leaf in param_sets:
    model = CatBoostRegressor(
        loss_function="RMSE",
        iterations=300,              # corto para búsqueda
        depth=depth,
        learning_rate=lr,
        l2_leaf_reg=l2,
        bagging_temperature=bt,
        min_data_in_leaf=min_leaf,
        random_seed=42,
        verbose=0
    )
    model.fit(
        X_tr, y_tr,
        cat_features=cat_features_idx,
        eval_set=(X_val, y_val),
        use_best_model=True,
        early_stopping_rounds=50
    )
    pred_val = model.predict(X_val)
    r2_val = r2_score(y_val, pred_val)
    if r2_val > best_r2:
        best_r2 = r2_val
        best_params = {
            "depth": depth,
            "learning_rate": lr,
            "l2_leaf_reg": l2,
            "bagging_temperature": bt,
            "min_data_in_leaf": min_leaf
        }
        best_model = model   # 👈 guardamos directamente el mejor modelo ya entrenado

print("✅ Mejores params (validación):", best_params, "R²_val:", round(best_r2, 4))
print("Best iteration (con early stopping):", best_model.get_best_iteration())

# --- Reentrenar en TODO el train con ese nº de iteraciones ---
final_model = CatBoostRegressor(
    loss_function="RMSE",
    iterations=best_model.get_best_iteration(),  # 👈 usamos el mismo número de iteraciones
    **best_params,
    random_seed=42,
    verbose=0
)
final_model.fit(
    X_train, y_train,
    cat_features=[X_train.columns.get_loc(c) for c in cat_features if c in X_train.columns]
)

# --- Evaluación en test ---
y_pred = final_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"CatBoost (final) R²: {r2:.4f}")
print(f"CatBoost (final) RMSE: {rmse:.4f}")


✅ Mejores params (validación): {'depth': 6, 'learning_rate': 0.08, 'l2_leaf_reg': 6, 'bagging_temperature': 0, 'min_data_in_leaf': 1} R²_val: 0.5623
Best iteration (con early stopping): 181
CatBoost (final) R²: 0.5712
CatBoost (final) RMSE: 0.4246


In [60]:
import pandas as pd

# Caso sintético con columnas seleccionadas
nuevo_jugador = pd.DataFrame([{
    'goals': 0,
    'touches': 60,
    'assists': 0,
    'result': 'draw',          # usa solo categorías que estén en tu df (ej. 'W', 'D', 'L')
    'duelos_ganados': 14,
    'poss_lost': 2,
    'passes_acc': 62,
    'minutesPlayed': 90,
    'interceptions': 10,
    'tackles': 11,
    'ycards': 0,
    'rcards': 0,
    'clearances': 10,
    'stop_shots': 10,
    'dangmistakes': 0,
    'fouls': 0

}])


In [43]:
def predecir_limitado(model, X, min_val=0, max_val=10):
    y_pred = model.predict(X)
    return np.clip(y_pred, min_val, max_val)

In [61]:
predecir_limitado(final_model, nuevo_jugador)

array([8.34794135])

In [62]:
final_model.save_model("catboost_defensas.cbm")
